# tidal-conductor — LoRA fine-tune for a local Brain (Colab, or any machine with a CUDA GPU)

Recipe: `training/README.md` in the repository.

**Status.** The author has run this notebook only on Google Colab and only with an earlier version of the
prompts. It has not been run with the prompts in this repository, and the local (non-Colab) path is the same
cells with different paths, not a separately tested procedure. Treat it as a starting point.

**On Colab**: Runtime → Change runtime type → **L4 GPU** (recommended; T4 works) → run the cells top to bottom.
Put the data (`train.jsonl`, `valid.jsonl`) in Google Drive under `tidal-conductor/data/` or upload it in the
left panel. **Outputs (GGUF + Modelfile) are saved to Drive under `tidal-conductor/`**; downloading multi-GB files
through the browser tends to stall, so let the Drive client sync them instead.

**On your own machine (CUDA GPU)**: `pip install unsloth jupyter`, start Jupyter from the repository's `training/`
directory and run the same cells. The data is read from `data/` and the outputs land in `fused/` (override with the
`TC_DATA_DIR` / `TC_OUT_DIR` environment variables). Apple Silicon (MLX) is not covered.

- The GGUF is exported **directly as q4_k_m** (no re-quantisation needed on the receiving machine)
- The Modelfile comes out with the production settings (temperature 0.6 / num_ctx 4096) — just `ollama create`
- **Before training**: run the parseBP gate (`haskell/check-cases.sh training/data/cases.ndjson`) and `pnpm build-dataset` so you train on the latest train/valid
- Do not pick a TPU runtime (Unsloth does not support it)

In [ ]:
%pip install -q unsloth

In [ ]:
# Where the data comes from and where the artifacts go.
# Colab: Google Drive (mounted here). Your own machine: run from the repository's training/ directory,
# then data/ and fused/ are the defaults; TC_DATA_DIR / TC_OUT_DIR override both anywhere.
import os
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/tidal-conductor"
    DATA_DIR = os.environ.get("TC_DATA_DIR", os.path.join(DRIVE_DIR, "data"))
    OUT_DIR = os.environ.get("TC_OUT_DIR", DRIVE_DIR)
else:
    DATA_DIR = os.environ.get("TC_DATA_DIR", "data")
    OUT_DIR = os.environ.get("TC_OUT_DIR", "fused")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print(f"colab={IN_COLAB} data={DATA_DIR} out={OUT_DIR}")

# train/valid: a file next to the notebook wins (Colab's upload panel), otherwise DATA_DIR
def data_file(name):
    path = name if os.path.exists(name) else os.path.join(DATA_DIR, name)
    assert os.path.exists(path), f"{name} is missing — put it in {DATA_DIR} (or next to the notebook)"
    return path
TRAIN_FILE, VALID_FILE = data_file("train.jsonl"), data_file("valid.jsonl")
print("train:", sum(1 for _ in open(TRAIN_FILE)), "samples / valid:", sum(1 for _ in open(VALID_FILE)), "samples")

In [ ]:
# Model setup.
# ⚠ Change the model family only through this cell: if the train_on_responses_only markers do not match
#   the family's chat template, the whole text is trained on without warning and the run is wasted
# ⚠ Qwen3 enables thinking (<think>) by default in its template — pin enable_thinking=False at every
#   stage (formatting, training, inference). The asserts below stop the run before training if it drifts
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"     # base used by the author; any instruct model of similar size works
# MODEL_NAME = "unsloth/Qwen3-1.7B"              # a thinking-capable alternative (lost on schema failures and latency in the author's comparison)
SAVE_NAME = "tidal-conductor-1.5b-r1"            # name in Ollama (bump per generation)

TEMPLATE_PARTS = {
    "qwen":  ("<|im_start|>user\n", "<|im_start|>assistant\n"),
    "llama": ("<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"),
}
FAMILY = "qwen" if "Qwen" in MODEL_NAME else "llama"
INSTRUCTION_PART, RESPONSE_PART = TEMPLATE_PARTS[FAMILY]
# Pin Qwen3 to non-thinking (Qwen2.5/llama ignore the unknown kwarg in their templates)
CHAT_KW = {"enable_thinking": False} if "Qwen3" in MODEL_NAME else {}
print(f"model={MODEL_NAME} family={FAMILY} save={SAVE_NAME} chat_kw={CHAT_KW}")

from unsloth import FastLanguageModel

MAX_SEQ = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Even with enable_thinking=False, Qwen3 inserts an empty thinking block at the start of the assistant turn
# (observed: "<|im_start|>assistant\n<think>\n\n</think>\n\ny").
# In production the JSON grammar forces the reply to start with "{", so strip the block from the training
# text and use plain ChatML with the JSON reply immediately after the assistant marker (distribution match)
EMPTY_THINK = "<think>\n\n</think>\n\n"
def strip_think(s):
    return s.replace(EMPTY_THINK, "")

# Guard: the markers must exist in the template (stop here if they drifted)
_probe = strip_think(tokenizer.apply_chat_template(
    [{"role": "user", "content": "x"}, {"role": "assistant", "content": "y"}],
    tokenize=False, **CHAT_KW,
))
assert INSTRUCTION_PART in _probe and RESPONSE_PART in _probe, \
    f"template mismatch: markers for {FAMILY} not found — check TEMPLATE_PARTS"
# Guard against thinking mode: after stripping, the content must follow the assistant marker directly
# (a non-empty thinking block or an unknown insertion stops here)
assert (RESPONSE_PART + "y") in _probe and "<think>" not in _probe, \
    f"the template inserts something before the assistant reply: {_probe!r}"
print("template markers / non-thinking check OK")

In [ ]:
# {"messages": [...]} → text with the chat template applied (thinking block stripped)
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": TRAIN_FILE, "valid": VALID_FILE})

def to_text(batch):
    return {"text": [
        strip_think(tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False, **CHAT_KW))
        for m in batch["messages"]
    ]}

ds = ds.map(to_text, batched=True, remove_columns=["messages"])
print(ds)
print(ds["train"][0]["text"][:300])
assert "<think>" not in ds["train"][0]["text"], "a thinking block leaked into the training text — check strip_think"

In [ ]:
# Training (the prompt part is excluded from the loss; only the assistant output is learned)
# 2 epochs by default (scales with the data size). If val loss is still falling at the end, rerun with 3.
# batch 16 assumes an L4 (24 GB) — drop to 8 on OOM
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["valid"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=1,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        output_dir="outputs",
        report_to="none",
        seed=42,
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part=INSTRUCTION_PART,
    response_part=RESPONSE_PART,
)
stats = trainer.train()
print(stats)

In [ ]:
# Smoke: generate from the first valid sample (eyeball that a plan JSON comes back).
# Strip the prompt the same way as in training (no thinking block)
import json
sample = json.loads(open(VALID_FILE).readline())
msgs = [m for m in sample["messages"] if m["role"] != "assistant"]
FastLanguageModel.for_inference(model)
text = strip_think(tokenizer.apply_chat_template(
    msgs, tokenize=False, add_generation_prompt=True, **CHAT_KW
))
inputs = tokenizer(text, return_tensors="pt").input_ids.to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=700, temperature=0.6)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# GGUF export (q4_k_m directly — q8_0 lost to memory bandwidth on an M1 in the author's tests) + save to OUT_DIR.
# If the progress bar freezes it is a stalled HF download — interrupt the cell and rerun to resume
model.save_pretrained_gguf(SAVE_NAME, tokenizer, quantization_method="q4_k_m")

import glob, re, shutil, os
gguf = glob.glob(f"{SAVE_NAME}*/**/*.gguf", recursive=True)[0]

# Modelfile to production settings: temperature 1.5 (Unsloth's default) is far too high for JSON output
mf_path = f"{SAVE_NAME}_gguf/Modelfile"
mf = open(mf_path).read()
mf = re.sub(r"PARAMETER temperature [0-9.]+", "PARAMETER temperature 0.6", mf)
mf = mf.rstrip("\n") + "\nPARAMETER num_ctx 4096\n"

# Put the generation in the file name: the converter names its output after the base model, so every
# generation would collide on Drive (an old GGUF was nearly deployed once). Rewrite FROM too, so
# `ollama create` works from the folder the files land in
dst_gguf = f"{SAVE_NAME}.Q4_K_M.gguf"
mf = re.sub(r"^FROM .*$", f"FROM ./{dst_gguf}", mf, flags=re.M)
# Qwen3 guard: leftovers of the thinking block in the Ollama template would mismatch the learned
# "JSON right after the assistant marker" distribution. Replace with plain ChatML automatically
# (stopping with an assert after 30 minutes of training would waste the run)
PLAIN_CHATML = """{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ .Content }}{{ if not $last }}<|im_end|>
{{ end }}
{{- end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
{{ end }}
{{- end }}"""
if "<think>" in mf:
    mf = re.sub(r'TEMPLATE """.*?"""', 'TEMPLATE """' + PLAIN_CHATML + '"""', mf, flags=re.S)
    print("⚠ thinking-block leftovers in the Modelfile → replaced with a plain ChatML template")
assert "<think>" not in mf, "<think> still present after the replacement — inspect the Modelfile"
open(mf_path, "w").write(mf)

# Rerun guard (Colab): if a previous run's flush_and_unmount detached Drive, the copy target is gone — remount
if IN_COLAB and not os.path.isdir(OUT_DIR):
    drive.mount("/content/drive")

# To OUT_DIR (on Colab that is Drive: multi-GB downloads through the browser tend to stall)
shutil.copy(gguf, os.path.join(OUT_DIR, dst_gguf))
shutil.copy(mf_path, os.path.join(OUT_DIR, f"Modelfile-{SAVE_NAME}"))
print(dst_gguf, f"+ Modelfile-{SAVE_NAME} → saved to {OUT_DIR}")
if IN_COLAB:
    drive.flush_and_unmount()
    print("flush done — wait for the Drive client to sync, then pick the files up")

## Afterwards (on the machine that runs Ollama)

Colab: once the Drive client has synced `tidal-conductor/`, copy the two files into `training/fused/`. Own machine:
they are already in `training/fused/` (or `TC_OUT_DIR`). Then create the model (the Modelfile is already adjusted,
no editing needed):

```sh
cp <where the Drive client put them>/*.gguf <...>/Modelfile-* training/fused/   # Colab only
cd training/fused
ollama create <SAVE_NAME> -f Modelfile-<SAVE_NAME>
cd ../..
```

**Acceptance: the four gates from `training/README.md`, Step 6** (close the browser first; run each twice and use the second run):

```sh
M=<SAVE_NAME>
AI_LOCAL_MODEL=$M pnpm api-smoke local 10                                                      # (1) baseline layout
AI_SMOKE_MANIFEST=manifests/smoke-alt.json AI_LOCAL_MODEL=$M pnpm api-smoke local 10           # (2) unseen layout (memorisation check)
AI_SMOKE_AVOID='t(3,8);t ~ t ~ t ~ t ~;0 1 0 ~ 2 3 0 ~ 4 5 ~ 6 ~ 7 6;t(5,16,<0 8>)' \
  AI_LOCAL_MODEL=$M pnpm api-smoke local 20                                                    # (3) veto compliance
AI_SMOKE_MANIFEST=manifests/smoke-kit.json AI_LOCAL_MODEL=$M pnpm api-smoke local 20           # (4) samples / nSet
```

Then `haskell/check-cases.sh ghci/api-smoke.ndjson` for the parseBP check. What to read: (1)(2) consistent 9/10 or better, (4) 19/20 or better,
parseBP ALL OK, latency median ≤ 4.5 s (second run). For (3), the Brain already discards a plan whose main pattern violates
the avoid list and strips a violating transition before returning, so count the "discarded for avoid violation" and
"partial degradation" lines in the output — fewer is better. The judgement is manual for now; the command's exit code
is 0 only when every round succeeded.
On Colab, copy `training/data/{train,valid}.jsonl` into `tidal-conductor/data/` on Drive before training.